In [64]:
import tensorflow as tf

In [65]:
# Download the Shakespeare dataset from TensorFlow
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

In [66]:
path

'/root/.keras/datasets/shakespeare.txt'

In [67]:
with open(path, "r", encoding="utf-8") as file:
    text = file.read()

In [68]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [69]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [70]:
# Create a tokenizer object
tokenizer = Tokenizer()

In [71]:
# Learn every unique word in the dataset
tokenizer.fit_on_texts([text])

In [72]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'i': 4,
 'of': 5,
 'you': 6,
 'my': 7,
 'a': 8,
 'that': 9,
 'in': 10,
 'is': 11,
 'not': 12,
 'for': 13,
 'with': 14,
 'me': 15,
 'it': 16,
 'be': 17,
 'your': 18,
 'his': 19,
 'but': 20,
 'this': 21,
 'he': 22,
 'have': 23,
 'as': 24,
 'thou': 25,
 'him': 26,
 'so': 27,
 'what': 28,
 'thy': 29,
 'will': 30,
 'by': 31,
 'no': 32,
 'all': 33,
 'king': 34,
 'we': 35,
 'shall': 36,
 'her': 37,
 'if': 38,
 'our': 39,
 'are': 40,
 'do': 41,
 'thee': 42,
 'lord': 43,
 'now': 44,
 'on': 45,
 'good': 46,
 'from': 47,
 'come': 48,
 'sir': 49,
 'or': 50,
 'which': 51,
 'more': 52,
 'then': 53,
 'at': 54,
 'o': 55,
 'would': 56,
 'was': 57,
 'they': 58,
 'how': 59,
 'well': 60,
 'here': 61,
 'she': 62,
 'than': 63,
 'their': 64,
 'them': 65,
 'duke': 66,
 'am': 67,
 'hath': 68,
 'say': 69,
 'let': 70,
 'when': 71,
 'one': 72,
 "i'll": 73,
 'go': 74,
 'love': 75,
 'were': 76,
 'may': 77,
 'us': 78,
 'make': 79,
 'like': 80,
 'upon': 81,
 'yet': 82,
 'richard': 83,

In [73]:
total_words = len(tokenizer.word_index) + 1

print(total_words)

12633


In [74]:
input_sequences = []

# Process one line at a time
for line in text.split("\n"):

    # Convert current line into integers
    token_list = tokenizer.texts_to_sequences([line])[0]

    # Ignore empty lines
    if len(token_list) < 1:
        continue

    # Create n-gram sequences
    for i in range(1, len(token_list)):
        n_gram = token_list[: i + 1]
        input_sequences.append(n_gram)

In [75]:
for seq in input_sequences[:10]:
    print(seq)

[88, 269]
[139, 35]
[139, 35, 969]
[139, 35, 969, 143]
[139, 35, 969, 143, 668]
[139, 35, 969, 143, 668, 127]
[139, 35, 969, 143, 668, 127, 15]
[139, 35, 969, 143, 668, 127, 15, 102]
[102, 102]
[88, 269]


In [76]:
print(tokenizer.index_word[88])
print(tokenizer.index_word[269])

first
citizen


In [77]:
for seq in input_sequences[:13]:
    print([tokenizer.index_word[i] for i in seq])

['first', 'citizen']
['before', 'we']
['before', 'we', 'proceed']
['before', 'we', 'proceed', 'any']
['before', 'we', 'proceed', 'any', 'further']
['before', 'we', 'proceed', 'any', 'further', 'hear']
['before', 'we', 'proceed', 'any', 'further', 'hear', 'me']
['before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak']
['speak', 'speak']
['first', 'citizen']
['you', 'are']
['you', 'are', 'all']
['you', 'are', 'all', 'resolved']


In [78]:
max_len = max(len(seq) for seq in input_sequences)

print("Maximum Sequence Length:", max_len)

Maximum Sequence Length: 16


In [79]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad all sequences to the same length
input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding="pre"
)

print(input_sequences[:5])
print(input_sequences.shape)

[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0  88 269]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0 139  35]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0 139  35 969]
 [  0   0   0   0   0   0   0   0   0   0   0   0 139  35 969 143]
 [  0   0   0   0   0   0   0   0   0   0   0 139  35 969 143 668]]
(171312, 16)


In [80]:
import numpy as np

# All words except the last one
X = input_sequences[:, :-1]

# Last word (next word to predict)
y = input_sequences[:, -1]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (171312, 15)
y Shape: (171312,)


In [81]:
print(X[:5])
print(y[:5])

[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0  88]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0 139]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0 139  35]
 [  0   0   0   0   0   0   0   0   0   0   0   0 139  35 969]
 [  0   0   0   0   0   0   0   0   0   0   0 139  35 969 143]]
[269  35 969 143 668]


In [82]:
# from tensorflow.keras.utils import to_categorical

# y = to_categorical(y, num_classes=total_words)

# print("One-Hot Encoded y Shape:", y.shape)

In [83]:
len(y)

171312

In [84]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Input

In [85]:
model = Sequential()

In [86]:
model.add(Input(shape=(max_len-1,)))
model.add(
    Embedding(
        input_dim=total_words,
        output_dim=100,
        input_length=max_len - 1 #15
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [87]:
# First LSTM
model.add(
    LSTM(
        150,
        return_sequences=True
    )
)

In [88]:
# Second LSTM
model.add(
    LSTM(100)
)

In [89]:
# Output Layer
model.add(
    Dense(
        total_words,
        activation="softmax"
    )
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 15, 100)        │     1,263,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 15, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12633)          │     1,275,933 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,790,233 (10.64 MB)

 Trainable params: 2,790,233 (10.64 MB)

 Non-trainable params: 0 (0.00 B)

In [90]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [91]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X,
    y,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - accuracy: 0.0356 - loss: 6.9537 - val_accuracy: 0.0365 - val_loss: 6.8354
Epoch 2/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.0488 - loss: 6.5125 - val_accuracy: 0.0508 - val_loss: 6.7495
Epoch 3/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.0636 - loss: 6.2787 - val_accuracy: 0.0635 - val_loss: 6.6880
Epoch 4/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.0838 - loss: 6.0668 - val_accuracy: 0.0757 - val_loss: 6.6487
Epoch 5/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.0937 - loss: 5.8912 - val_accuracy: 0.0797 - val_loss: 6.6232
Epoch 6/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.1001 - loss: 5.7499 - val_accuracy: 0.0813 - val_loss: 6.6410
Epoch 7/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.1054 - loss: 5.6266 - val_accuracy: 0.0820 - val_loss: 6.6897
Epoch 8/50
1071/1071 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.1100 -

In [92]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(seed_text):
    # Convert input text to token IDs
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # Pad to the required length
    token_list = pad_sequences(
        [token_list],
        maxlen=max_len - 1,
        padding="pre"
    )

    # Predict probabilities
    prediction = model.predict(token_list, verbose=0)

    # Get the word ID with highest probability
    predicted_id = np.argmax(prediction, axis=1)[0]

    # Convert ID back to word
    for word, index in tokenizer.word_index.items():
        if index == predicted_id:
            return word

In [93]:
print(predict_next_word("before we"))
print(predict_next_word("to be"))
print(predict_next_word("shall"))

have
a
be


In [94]:
predict_next_word("Caius Marcius")

'is'